In [1]:
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

def prepare_data(df, is_train=True):
    data = df.copy()
    
    ids = data['id'] if 'id' in data.columns else None
    if 'id' in data.columns:
        data = data.drop('id', axis=1)
    
    if is_train and 'Status' in data.columns:
        y = data['Status'].copy()
        data = data.drop('Status', axis=1)
    else:
        y = None
    
    categorical_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
    
    for col in categorical_cols:
        if col in data.columns:
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col].astype(str))
    
    return data, y, ids

X_train, y_train, _ = prepare_data(train, is_train=True)
X_test, _, test_ids = prepare_data(test, is_train=False)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Classes:", y_train.unique())

X_train shape: (15000, 18)
X_test shape: (10000, 18)
Classes: <StringArray>
['D', 'C', 'CL']
Length: 3, dtype: str


In [3]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 1e-5, 10, log=True),
        'one_hot_max_size': trial.suggest_int('one_hot_max_size', 2, 10),
    }
    
    # Используем 5-fold cross-validation для большей стабильности
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]
        
        model = CatBoostClassifier(
            **params,
            loss_function='MultiClass',
            eval_metric='MultiClass',
            verbose=False,
            random_seed=42,
            early_stopping_rounds=50
        )
        
        model.fit(
            X_fold_train, y_fold_train,
            eval_set=(X_fold_val, y_fold_val),
            verbose=False
        )
        
        pred = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, pred)
        scores.append(score)
    
    return np.mean(scores)

In [4]:
study = optuna.create_study(direction='minimize', study_name='catboost_optimization')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best trial:")
print(f"Log Loss: {study.best_value:.4f}")
print("Best params:", study.best_params)

[I 2026-03-15 18:29:26,192] A new study created in memory with name: catboost_optimization


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-15 18:29:31,263] Trial 0 finished with value: 0.39964407483348 and parameters: {'iterations': 814, 'depth': 4, 'learning_rate': 0.01010709945403885, 'l2_leaf_reg': 4.240058890965376, 'border_count': 198, 'bagging_temperature': 0.8872496219578124, 'random_strength': 0.00010335018386554718, 'one_hot_max_size': 4}. Best is trial 0 with value: 0.39964407483348.
[I 2026-03-15 18:29:34,794] Trial 1 finished with value: 0.3752847451360465 and parameters: {'iterations': 507, 'depth': 4, 'learning_rate': 0.11296508201968962, 'l2_leaf_reg': 3.758149670419437, 'border_count': 177, 'bagging_temperature': 0.9041002272786558, 'random_strength': 0.0004088140596134894, 'one_hot_max_size': 5}. Best is trial 1 with value: 0.3752847451360465.
[I 2026-03-15 18:29:40,260] Trial 2 finished with value: 0.3933082536840002 and parameters: {'iterations': 709, 'depth': 5, 'learning_rate': 0.01662581442444909, 'l2_leaf_reg': 10.160865579114745, 'border_count': 55, 'bagging_temperature': 0.9859648345355

In [5]:
best_model = CatBoostClassifier(
    **study.best_params,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    verbose=100,
    random_seed=42
)

best_model.fit(X_train, y_train)

# Оценка на валидации (разделим данные)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
val_pred = best_model.predict_proba(X_val)
val_loss = log_loss(y_val, val_pred)
print(f"Final CatBoost Log Loss on validation: {val_loss:.4f}")

0:	learn: 0.9777216	total: 2.86ms	remaining: 1.7s
100:	learn: 0.3730781	total: 160ms	remaining: 781ms
200:	learn: 0.3498146	total: 332ms	remaining: 649ms
300:	learn: 0.3333019	total: 499ms	remaining: 485ms
400:	learn: 0.3203280	total: 654ms	remaining: 315ms
500:	learn: 0.3088200	total: 831ms	remaining: 154ms
593:	learn: 0.2989734	total: 991ms	remaining: 0us
Final CatBoost Log Loss on validation: 0.3111


In [6]:
test_pred = best_model.predict_proba(X_test)

submission = pd.DataFrame({
    'id': test_ids,
    'Status_C': test_pred[:, 0],
    'Status_CL': test_pred[:, 1],
    'Status_D': test_pred[:, 2]
})

submission.to_csv('../data/submission.csv', index=False)
print("Submission saved to ../data/submission.csv")
print(submission.head())

Submission saved to ../data/submission.csv
      id  Status_C  Status_CL  Status_D
0  15000  0.948656   0.029408  0.021935
1  15001  0.984562   0.002411  0.013027
2  15002  0.454882   0.062458  0.482659
3  15003  0.044908   0.030593  0.924499
4  15004  0.921606   0.008694  0.069700


In [7]:
print("Prediction statistics:")
print(f"Status_C mean: {test_pred[:, 0].mean():.4f}")
print(f"Status_CL mean: {test_pred[:, 1].mean():.4f}")
print(f"Status_D mean: {test_pred[:, 2].mean():.4f}")

Prediction statistics:
Status_C mean: 0.6777
Status_CL mean: 0.0243
Status_D mean: 0.2980
